# Notebook 3 — Train / Validation / Test Split

## Load the labeled table from 02_labels.ipynb

In [1]:
import pandas as pd

df = pd.read_csv("../artifacts/02_labeled_table.csv", parse_dates=["order_purchase_timestamp"])

df.shape

(96470, 22)

**Note:** Loaded 96,470 rows and 22 columns — matches the labeled table 
saved in Notebook 2 (02_labels.ipynb)

## Check the date range of the data

In [3]:
print(df["order_purchase_timestamp"].min())
print(df["order_purchase_timestamp"].max())

2016-09-15 12:16:38
2018-08-29 15:00:37


**Note:** Data spans from September 2016 to August 2018 — about 2 years. 
This confirms the data is suitable for a time-based split: earlier orders 
will form the training set, and the most recent orders will form the test set.

## Sort by date and split into train / validation / test

In [4]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

print("train:", train.shape, "|", train["order_purchase_timestamp"].min(), "-", train["order_purchase_timestamp"].max())
print("val:  ", val.shape, "|", val["order_purchase_timestamp"].min(), "-", val["order_purchase_timestamp"].max())
print("test: ", test.shape, "|", test["order_purchase_timestamp"].min(), "-", test["order_purchase_timestamp"].max())

train: (67529, 22) | 2016-09-15 12:16:38 - 2018-04-15 20:12:35
val:   (14470, 22) | 2018-04-15 20:17:11 - 2018-06-21 08:29:29
test:  (14471, 22) | 2018-06-21 08:41:07 - 2018-08-29 15:00:37


**Note:** Split into 67,529 train / 14,470 validation / 14,471 test rows 
(70% / 15% / 15%), with no overlap in time periods — train covers the 
earliest orders, test covers the most recent ones. This matches how the 
model will be used in production: trained on past data, evaluated on 
future data.

## Check label balance across splits

In [5]:
print("train late %:", train["is_late"].mean())
print("val late %:  ", val["is_late"].mean())
print("test late %: ", test["is_late"].mean())

train late %: 0.09027232744450532
val late %:   0.053420870767104355
test late %:  0.06613226452905811


**Note:** Late-delivery rate differs across splits: 9.0% in train, 5.3% in 
validation, 6.6% in test. This is expected with a time-based split (not a 
problem to fix) — it likely reflects real changes in delivery performance 
over time. This is worth exploring further in the EDA notebook (e.g., 
does the late rate trend down over the months?).

## Save train, validation, and test files

In [6]:
train.to_csv("../artifacts/03_train.csv", index=False)
val.to_csv("../artifacts/03_val.csv", index=False)
test.to_csv("../artifacts/03_test.csv", index=False)

print("Saved train, val, test files.")

Saved train, val, test files.
